# Urejanje podatkov in združevanje tabel

Sestavili bomo tabeli `studenti` in `studenti_ocene` iz [prejšnjega zvezka](urejanje-podatkov.ipynb). Pri tem bomo spoznali metode knjižnice `pandas` za urejanje podatkov in združevanje tabel, ki ustreza operaciji `JOIN` v jeziku SQL.

Najprej s funkcijo [`read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) ponovno uvozimo vse liste preglednice Excel. Vsaki tabeli dodamo ime lista in tabeli združimo s funkcijo [`concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html):

In [1]:
import pandas as pd

vse_pt = pd.read_excel(
    "https://kt.ijs.si/~ljupco/lectures/papvp-2324/rezultati-kolokvijev.xlsx",
    sheet_name=None
)
for ime_lista in vse_pt:
    vse_pt[ime_lista]["kolokvij"] = ime_lista
pt = pd.concat(vse_pt.values())

## Razdruževanje vrednosti

Stolpec `spol in starost` vsebuje vrednosti dveh spremenljivk. Razdelimo jih z metodo [`str.split`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html), ki je prek atributa `str` dostopna pri zaporedju besedilnih vrednosti:

In [2]:
print(pt["spol in starost"].str.split("-"))

0    [ž, 19]
1    [m, 20]
2    [m, 19]
3    [ž, 20]
4    [m, 21]
0    [ž, 19]
1    [ž, 21]
2    [m, 19]
3    [ž, 20]
4    [m, 21]
Name: spol in starost, dtype: object


Rezultat je zaporedje seznamov. Z argumentom `expand=True` namesto tega dobimo podatkovno tabelo z ločenimi stolpci:

In [3]:
print(pt["spol in starost"].str.split("-", expand=True))

   0   1
0  ž  19
1  m  20
2  m  19
3  ž  20
4  m  21
0  ž  19
1  ž  21
2  m  19
3  ž  20
4  m  21


Rezultat s prirejanjem shranimo v dva nova stolpca tabele `pt`:

In [4]:
pt[["spol", "starost"]] = pt["spol in starost"].str.split("-", expand=True)
print(pt)

      ime  telefon spol in starost  prvi semester  drugi semester  \
0     Ana      431            ž-19             80              97   
1  Branko      720            m-20             78              74   
2   David      141            m-19             75              68   
3     Eva      210            ž-20             63              82   
4   Franc      592            m-21             95              82   
0     Ana      431            ž-19             82              95   
1  Cvetka      761            ž-21             63              87   
2   David      141            m-19             92              81   
3     Eva      210            ž-20             82              63   
4   Franc      592            m-21             99              54   

         kolokvij spol starost  
0   prvi kolokvij    ž      19  
1   prvi kolokvij    m      20  
2   prvi kolokvij    m      19  
3   prvi kolokvij    ž      20  
4   prvi kolokvij    m      21  
0  drugi kolokvij    ž      19  
1  drugi 

## Tvorjenje in urejanje tabele `studenti`

Z indeksiranjem prek atributa [`loc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) izberemo stolpce za tabelo `studenti`:

In [5]:
studenti = pt.loc[:, ["ime", "spol", "starost", "telefon"]]
print(studenti)

      ime spol starost  telefon
0     Ana    ž      19      431
1  Branko    m      20      720
2   David    m      19      141
3     Eva    ž      20      210
4   Franc    m      21      592
0     Ana    ž      19      431
1  Cvetka    ž      21      761
2   David    m      19      141
3     Eva    ž      20      210
4   Franc    m      21      592


Tabela vsebuje ponovljene podatke o nekaterih študentih. Najprej poskusimo izbrati šest vrstic po njihovih oznakah:

In [6]:
print(pt.loc[[0, 1, 2, 3, 4, 1], ["ime", "spol", "starost", "telefon"]])

      ime spol starost  telefon
0     Ana    ž      19      431
0     Ana    ž      19      431
1  Branko    m      20      720
1  Cvetka    ž      21      761
2   David    m      19      141
2   David    m      19      141
3     Eva    ž      20      210
3     Eva    ž      20      210
4   Franc    m      21      592
4   Franc    m      21      592
1  Branko    m      20      720
1  Cvetka    ž      21      761


Poskus ni uspel, saj se oznake vrstic ponavljajo. Z atributom [`iloc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) lahko vrstice izberemo po položajih:

In [7]:
print(studenti.iloc[[0, 1, 2, 3, 4, 6]])

      ime spol starost  telefon
0     Ana    ž      19      431
1  Branko    m      20      720
2   David    m      19      141
3     Eva    ž      20      210
4   Franc    m      21      592
1  Cvetka    ž      21      761


Pri tem moramo položaje vrstic določiti ročno, kar za velike tabele ni praktično.

Metoda [`drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) odstrani podvojene vrstice in vrne novo tabelo:

In [8]:
studenti = studenti.drop_duplicates()
print(studenti)

      ime spol starost  telefon
0     Ana    ž      19      431
1  Branko    m      20      720
2   David    m      19      141
3     Eva    ž      20      210
4   Franc    m      21      592
1  Cvetka    ž      21      761


Vrstice uredimo naraščajoče po vrednostih dimenzijske spremenljivke `ime`. Uporabimo metodo [`sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html). Z argumentom `by` določimo ime stolpca ali seznam imen stolpcev za urejanje:

In [9]:
studenti = studenti.sort_values(by="ime")
print(studenti)

      ime spol starost  telefon
0     Ana    ž      19      431
1  Branko    m      20      720
1  Cvetka    ž      21      761
2   David    m      19      141
3     Eva    ž      20      210
4   Franc    m      21      592


Z atributom [`dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) preverimo podatkovne tipe stolpcev:

In [10]:
print(studenti.dtypes)

ime        object
spol       object
starost    object
telefon     int64
dtype: object


Stolpec `spol` naj bo kategoričnega tipa, stolpec `starost` pa celoštevilskega tipa. Za pretvorbo uporabimo metodo [`astype`](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) in razred [`CategoricalDtype`](https://pandas.pydata.org/docs/reference/api/pandas.CategoricalDtype.html). Kategorije dobimo z metodo [`drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.Series.drop_duplicates.html) za zaporedje:

In [11]:
from pandas.api.types import CategoricalDtype

studenti["spol"] = studenti["spol"].astype(CategoricalDtype(studenti["spol"].drop_duplicates()))
studenti["starost"] = studenti["starost"].astype("int64")

print(studenti.dtypes)

ime          object
spol       category
starost       int64
telefon       int64
dtype: object


Poglejmo urejeno tabelo `studenti`:

In [12]:
print(studenti)

      ime spol  starost  telefon
0     Ana    ž       19      431
1  Branko    m       20      720
1  Cvetka    ž       21      761
2   David    m       19      141
3     Eva    ž       20      210
4   Franc    m       21      592


## Tvorjenje in urejanje tabele `studenti_ocene`

Izberimo podatke o dosežkih študentov:

In [13]:
studenti_ocene = pt[["ime", "prvi semester", "drugi semester", "kolokvij"]]
print(studenti_ocene)

      ime  prvi semester  drugi semester        kolokvij
0     Ana             80              97   prvi kolokvij
1  Branko             78              74   prvi kolokvij
2   David             75              68   prvi kolokvij
3     Eva             63              82   prvi kolokvij
4   Franc             95              82   prvi kolokvij
0     Ana             82              95  drugi kolokvij
1  Cvetka             63              87  drugi kolokvij
2   David             92              81  drugi kolokvij
3     Eva             82              63  drugi kolokvij
4   Franc             99              54  drugi kolokvij


Vrednosti `prvi kolokvij` in `drugi kolokvij` z metodo [`replace`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html) nadomestimo s celima številoma:

In [14]:
studenti_ocene = studenti_ocene.replace({
    "prvi kolokvij": 1,
    "drugi kolokvij": 2
})
print(studenti_ocene)

      ime  prvi semester  drugi semester  kolokvij
0     Ana             80              97         1
1  Branko             78              74         1
2   David             75              68         1
3     Eva             63              82         1
4   Franc             95              82         1
0     Ana             82              95         2
1  Cvetka             63              87         2
2   David             92              81         2
3     Eva             82              63         2
4   Franc             99              54         2


Imeni stolpcev `prvi semester` in `drugi semester` morata postati vrednosti novega stolpca `semester`. To pretvorbo opravimo z metodo [`melt`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html):

  * `id_vars` določa stolpce, katerih vrednosti ohranimo ob posamezni meritvi;
  * `value_vars` določa stolpce z meritvami, ki jih združimo v en stolpec. Privzeto so to vsi stolpci, ki niso navedeni v `id_vars`;
  * `var_name` določa ime novega stolpca, ki prevzame imena stolpcev iz `value_vars`;
  * `value_name` določa ime novega stolpca z vrednostmi meritev.

Tabelo podamo kot objekt, na katerem pokličemo metodo:

In [15]:
studenti_ocene = studenti_ocene.melt(
    id_vars=["ime", "kolokvij"],
    value_vars=["prvi semester", "drugi semester"],
    var_name="semester",
    value_name="rezultat"
)
print(studenti_ocene)

       ime  kolokvij        semester  rezultat
0      Ana         1   prvi semester        80
1   Branko         1   prvi semester        78
2    David         1   prvi semester        75
3      Eva         1   prvi semester        63
4    Franc         1   prvi semester        95
5      Ana         2   prvi semester        82
6   Cvetka         2   prvi semester        63
7    David         2   prvi semester        92
8      Eva         2   prvi semester        82
9    Franc         2   prvi semester        99
10     Ana         1  drugi semester        97
11  Branko         1  drugi semester        74
12   David         1  drugi semester        68
13     Eva         1  drugi semester        82
14   Franc         1  drugi semester        82
15     Ana         2  drugi semester        95
16  Cvetka         2  drugi semester        87
17   David         2  drugi semester        81
18     Eva         2  drugi semester        63
19   Franc         2  drugi semester        54


Z metodo [`replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) za zaporedje nadomestimo še vrednosti stolpca `semester`:

In [16]:
studenti_ocene["semester"] = studenti_ocene["semester"].replace({
    "prvi semester": 1,
    "drugi semester": 2
})
print(studenti_ocene)

       ime  kolokvij  semester  rezultat
0      Ana         1         1        80
1   Branko         1         1        78
2    David         1         1        75
3      Eva         1         1        63
4    Franc         1         1        95
5      Ana         2         1        82
6   Cvetka         2         1        63
7    David         2         1        92
8      Eva         2         1        82
9    Franc         2         1        99
10     Ana         1         2        97
11  Branko         1         2        74
12   David         1         2        68
13     Eva         1         2        82
14   Franc         1         2        82
15     Ana         2         2        95
16  Cvetka         2         2        87
17   David         2         2        81
18     Eva         2         2        63
19   Franc         2         2        54


Stolpce postavimo v želeni vrstni red, vrstice pa uredimo naraščajoče po dimenzijskih spremenljivkah:

In [17]:
studenti_ocene = studenti_ocene[["ime", "semester", "kolokvij", "rezultat"]]
studenti_ocene = studenti_ocene.sort_values(by=["ime", "semester", "kolokvij"])

print(studenti_ocene)

       ime  semester  kolokvij  rezultat
0      Ana         1         1        80
5      Ana         1         2        82
10     Ana         2         1        97
15     Ana         2         2        95
1   Branko         1         1        78
11  Branko         2         1        74
6   Cvetka         1         2        63
16  Cvetka         2         2        87
2    David         1         1        75
7    David         1         2        92
12   David         2         1        68
17   David         2         2        81
3      Eva         1         1        63
8      Eva         1         2        82
13     Eva         2         1        82
18     Eva         2         2        63
4    Franc         1         1        95
9    Franc         1         2        99
14   Franc         2         1        82
19   Franc         2         2        54


Preverimo še podatkovne tipe:

In [18]:
print(studenti_ocene.dtypes)

ime         object
semester     int64
kolokvij     int64
rezultat     int64
dtype: object


Podatkovni tipi ustrezajo zapisanim vrednostim.

Tabela ne vsebuje vseh kombinacij vrednosti dimenzijskih spremenljivk. Ne vemo na primer, kakšen je bil Brankov dosežek na drugem kolokviju v prvem semestru. Predpostavimo, da h kolokviju ni pristopil. To lahko izrecno zapišemo z vrstico z vrednostmi `Branko`, `1`, `2` in manjkajočim rezultatom, označenim z `NaN`.

Dopolnjevanje vseh kombinacij ni zahteva dogovora o urejenih podatkih. V tem primeru ga izberemo, da ločimo manjkajoče rezultate od manjkajočih vrstic. Pri tem predpostavljamo, da bi vsak študent lahko opravljal vse štiri kolokvije.

## Združevanje tabel in poizvedovalni jezik SQL

SQL (angl. _Structured Query Language_) je jezik za poizvedovanje po podatkih v relacijskih podatkovnih bazah. Operacija `JOIN` povezuje vrstice različnih tabel po vrednostih skupnih stolpcev.

V knjižnici `pandas` tabeli združimo z metodo [`merge`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html). Za vsako kombinacijo vrednosti ključnih stolpcev poveže vse ustrezne vrstice leve in desne tabele. Če se ključ v obeh tabelah ponavlja, dobimo vse kombinacije pripadajočih vrstic. Skupni ključni stolpci se pri združevanju z `on` pojavijo enkrat; drugi istoimenski stolpci dobijo priponi `_x` in `_y`.

Metoda ima naslednje argumente:

  * `right` podaja desno tabelo; levo tabelo podamo kot objekt, na katerem pokličemo metodo;
  * `on` določa ime ali seznam imen ključnih stolpcev. Če ga izpustimo, metoda privzeto uporabi vsa skupna imena stolpcev;
  * `how` določa način združevanja:
    * `"left"` ohrani vse vrstice leve tabele in jim doda ujemajoče se vrstice desne;
    * `"right"` ohrani vse vrstice desne tabele in jim doda ujemajoče se vrstice leve;
    * `"inner"` ohrani le kombinacije vrstic z ujemajočimi se ključi;
    * `"outer"` poleg ujemajočih se vrstic ohrani tudi vrstice brez para iz obeh tabel;
    * `"cross"` tvori kartezični produkt vseh vrstic; pri tem ne podamo argumenta `on`.

Kjer pri združevanju vrstica nima para, so dodane vrednosti manjkajoče.

Ustvarimo dve podatkovni tabeli s konstruktorjem [`DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html). Stolpce podamo kot zaporedja razreda [`Series`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html). Za ključ uporabimo stolpec `id`:

In [19]:
dt_x = pd.DataFrame({
    "id": pd.Series([1, 1, 2, 2, 3]),
    "vrednost": pd.Series(["x11", "x12", "x21", "x22", "x3"])
})
print(dt_x)

   id vrednost
0   1      x11
1   1      x12
2   2      x21
3   2      x22
4   3       x3


In [20]:
dt_y = pd.DataFrame({
    "id": pd.Series([1, 2, 2, 4, 4]),
    "vrednost": pd.Series(["y1", "y21", "y22", "y41", "y42"])
})
print(dt_y)

   id vrednost
0   1       y1
1   2      y21
2   2      y22
3   4      y41
4   4      y42


Levo združevanje ohrani vse vrstice tabele `dt_x`. Vrstica z `id=3` v desni tabeli nima para, zato je njena vrednost v dodanem stolpcu manjkajoča. Pri `id=2` se povežeta dve levi in dve desni vrstici, zato dobimo štiri kombinacije:

In [21]:
print(dt_x.merge(dt_y, on="id", how="left"))

   id vrednost_x vrednost_y
0   1        x11         y1
1   1        x12         y1
2   2        x21        y21
3   2        x21        y22
4   2        x22        y21
5   2        x22        y22
6   3         x3        NaN


Desno združevanje ohrani vse vrstice desne tabele:

In [22]:
print(dt_x.merge(dt_y, on="id", how="right"))

   id vrednost_x vrednost_y
0   1        x11         y1
1   1        x12         y1
2   2        x21        y21
3   2        x22        y21
4   2        x21        y22
5   2        x22        y22
6   4        NaN        y41
7   4        NaN        y42


Poglejmo še notranje (`inner`) in zunanje (`outer`) združevanje:

In [23]:
print(dt_x.merge(dt_y, on="id", how="inner"))

   id vrednost_x vrednost_y
0   1        x11         y1
1   1        x12         y1
2   2        x21        y21
3   2        x21        y22
4   2        x22        y21
5   2        x22        y22


In [24]:
print(dt_x.merge(dt_y, on="id", how="outer"))

   id vrednost_x vrednost_y
0   1        x11         y1
1   1        x12         y1
2   2        x21        y21
3   2        x21        y22
4   2        x22        y21
5   2        x22        y22
6   3         x3        NaN
7   4        NaN        y41
8   4        NaN        y42


V tem primeru notranje združevanje nima manjkajočih vrednosti, saj vhodni tabeli nimata manjkajočih podatkov in ohranimo le vrstice z ujemajočimi se ključi. Zunanje združevanje ohrani tudi vrstico z `id=3` in dve vrstici z `id=4`, zato doda tri manjkajoče vrednosti.

Združevanje z `how="cross"` vrne kartezični produkt vrstic obeh tabel, brez primerjanja vrednosti ključev:

In [25]:
print(dt_x.merge(dt_y, how="cross"))

    id_x vrednost_x  id_y vrednost_y
0      1        x11     1         y1
1      1        x11     2        y21
2      1        x11     2        y22
3      1        x11     4        y41
4      1        x11     4        y42
5      1        x12     1         y1
6      1        x12     2        y21
7      1        x12     2        y22
8      1        x12     4        y41
9      1        x12     4        y42
10     2        x21     1         y1
11     2        x21     2        y21
12     2        x21     2        y22
13     2        x21     4        y41
14     2        x21     4        y42
15     2        x22     1         y1
16     2        x22     2        y21
17     2        x22     2        y22
18     2        x22     4        y41
19     2        x22     4        y42
20     3         x3     1         y1
21     3         x3     2        y21
22     3         x3     2        y22
23     3         x3     4        y41
24     3         x3     4        y42


## Dopolnjevanje tabele `studenti_ocene`

Za vsako kombinacijo vrednosti dimenzijskih spremenljivk želimo imeti vrstico. Najprej sestavimo tabele domen posameznih dimenzijskih spremenljivk, nato pa njihov kartezični produkt:

In [26]:
d_ime = pd.DataFrame({
    "ime": studenti_ocene["ime"].drop_duplicates()
})
d_semester = pd.DataFrame({
    "semester": studenti_ocene["semester"].drop_duplicates()
})
d_kolokvij = pd.DataFrame({
    "kolokvij": studenti_ocene["kolokvij"].drop_duplicates()
})

d_vse = d_ime.merge(d_semester, how="cross").merge(d_kolokvij, how="cross")
print(d_vse)

       ime  semester  kolokvij
0      Ana         1         1
1      Ana         1         2
2      Ana         2         1
3      Ana         2         2
4   Branko         1         1
5   Branko         1         2
6   Branko         2         1
7   Branko         2         2
8   Cvetka         1         1
9   Cvetka         1         2
10  Cvetka         2         1
11  Cvetka         2         2
12   David         1         1
13   David         1         2
14   David         2         1
15   David         2         2
16     Eva         1         1
17     Eva         1         2
18     Eva         2         1
19     Eva         2         2
20   Franc         1         1
21   Franc         1         2
22   Franc         2         1
23   Franc         2         2


Tabela vseh kombinacij ima 24 vrstic: šest študentov, dva semestra in dva kolokvija v semestru.

Z levim združevanjem vsaki kombinaciji dodamo doseženi rezultat. Ključne stolpce navedemo izrecno:

In [27]:
studenti_ocene = d_vse.merge(
    studenti_ocene, on=["ime", "semester", "kolokvij"], how="left"
)
print(studenti_ocene)

       ime  semester  kolokvij  rezultat
0      Ana         1         1      80.0
1      Ana         1         2      82.0
2      Ana         2         1      97.0
3      Ana         2         2      95.0
4   Branko         1         1      78.0
5   Branko         1         2       NaN
6   Branko         2         1      74.0
7   Branko         2         2       NaN
8   Cvetka         1         1       NaN
9   Cvetka         1         2      63.0
10  Cvetka         2         1       NaN
11  Cvetka         2         2      87.0
12   David         1         1      75.0
13   David         1         2      92.0
14   David         2         1      68.0
15   David         2         2      81.0
16     Eva         1         1      63.0
17     Eva         1         2      82.0
18     Eva         2         1      82.0
19     Eva         2         2      63.0
20   Franc         1         1      95.0
21   Franc         1         2      99.0
22   Franc         2         1      82.0
23   Franc      

Tabela `studenti_ocene` zdaj vsebuje vseh 24 kombinacij dimenzijskih vrednosti. Pri kombinacijah brez zapisanega dosežka je rezultat manjkajoč. Tabeli `studenti` in `studenti_ocene` sta pripravljeni za analizo in vizualizacijo.

Vaje in naloge so zbrane v zvezku [Naloge za vaje](urejanje-naloge.ipynb).